# Encounter / Visit data analysis (Snowflake, read-only)

One row = one visit. **Read-only**: `SELECT` / `DESCRIBE` only.

**Cell format:** analysis cells are **SQL cells**, so each result shows Table / Chart / Pivot and the **download** button.

**Grain:** `EncounterId/VisitId` should be unique. `Member/PatientId` is **not** unique here — one patient can have many visits.

## 1. Active session

In [ ]:
import pandas as pd

from snowflake.snowpark.context import get_active_session

session = get_active_session()
session

## 2. Config (change names ONLY here)

SQL cells read flat strings (`T`, `C_ENC`, `C_TYPE`, ...). `EncounterId/VisitId` and `Encounter/Visit Date` need quotes.

In [ ]:
DATABASE_NAME = "ATTR"
SCHEMA_NAME = "PUBLIC"
TABLE_NAME = "ENCOUNTER"   # try ENCOUNTER, VISIT, ENCOUNTER_VISIT

QUOTE_DATABASE = False
QUOTE_SCHEMA = False
QUOTE_TABLE = False
QUOTE_COLUMNS = True

COL = {
    "encounter_id": "EncounterId/VisitId",
    "patient_id": "Member/PatientId",
    "facility_id": "FacilityId",
    "physician_id": "PhysicianId",
    "visit_date": "Encounter/Visit Date",
    "visit_type": "Type",
    "status": "Status",
}


def sf_ident(name, quoted):
    if quoted:
        return '"' + str(name).replace('"', '""') + '"'
    return str(name)


def col(key):
    return sf_ident(COL[key], QUOTE_COLUMNS)


DB = sf_ident(DATABASE_NAME, QUOTE_DATABASE)
T = ".".join(
    [
        DB,
        sf_ident(SCHEMA_NAME, QUOTE_SCHEMA),
        sf_ident(TABLE_NAME, QUOTE_TABLE),
    ]
)

C_ENC = col("encounter_id")
C_PT = col("patient_id")
C_FAC = col("facility_id")
C_PHY = col("physician_id")
C_DATE = col("visit_date")
C_TYPE = col("visit_type")
C_STATUS = col("status")

print(f"T = {T}")
print(f"C_ENC = {C_ENC}")
print(f"C_PT = {C_PT}")
print(f"C_FAC = {C_FAC}")
print(f"C_PHY = {C_PHY}")
print(f"C_DATE = {C_DATE}")
print(f"C_TYPE = {C_TYPE}")
print(f"C_STATUS = {C_STATUS}")

## 3. Find the table (only if the name or schema is wrong)

In [ ]:
SELECT
    CURRENT_ROLE() AS ROLE,
    CURRENT_WAREHOUSE() AS WAREHOUSE,
    CURRENT_DATABASE() AS DATABASE,
    CURRENT_SCHEMA() AS SCHEMA;

In [ ]:
SELECT
    TABLE_CATALOG,
    TABLE_SCHEMA,
    TABLE_NAME,
    ROW_COUNT,
    BYTES
FROM {{DB}}.INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
  AND (
        UPPER(TABLE_NAME) LIKE '%ENCOUNTER%'
     OR UPPER(TABLE_NAME) LIKE '%VISIT%'
  )
ORDER BY TABLE_SCHEMA, TABLE_NAME;

## 4. Table shape and first 10 rows

In [ ]:
DESCRIBE TABLE {{T}};

In [ ]:
SELECT *
FROM {{T}}
LIMIT 10;

## 5. Volume and uniqueness

Expected: unique encounters close to row count, and fewer unique patients (repeat visits).

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_FAC}}) AS UNIQUE_FACILITIES,
    COUNT(DISTINCT {{C_PHY}}) AS UNIQUE_PHYSICIANS,
    COUNT(*) - COUNT(DISTINCT {{C_ENC}}) AS EXTRA_ROWS_VS_UNIQUE_ENCOUNTERS,
    ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT {{C_PT}}), 0), 2) AS AVG_VISITS_PER_PATIENT
FROM {{T}};

## 6. Completeness (nulls)

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    SUM(IFF({{C_ENC}} IS NULL, 1, 0)) AS NULL_ENCOUNTER_ID,
    SUM(IFF({{C_PT}} IS NULL, 1, 0)) AS NULL_PATIENT_ID,
    SUM(IFF({{C_FAC}} IS NULL, 1, 0)) AS NULL_FACILITY_ID,
    SUM(IFF({{C_PHY}} IS NULL, 1, 0)) AS NULL_PHYSICIAN_ID,
    SUM(IFF({{C_DATE}} IS NULL, 1, 0)) AS NULL_VISIT_DATE,
    SUM(IFF({{C_TYPE}} IS NULL, 1, 0)) AS NULL_TYPE,
    SUM(IFF({{C_STATUS}} IS NULL, 1, 0)) AS NULL_STATUS
FROM {{T}};

## 7. Visit date range and distribution

This is the **visit calendar date**, not birth date. Recency is today minus visit date.

In [ ]:
SELECT
    CURRENT_DATE() AS TODAY,
    MIN({{C_DATE}}) AS MIN_VISIT_DATE,
    MAX({{C_DATE}}) AS MAX_VISIT_DATE,
    DATEDIFF('day', MIN({{C_DATE}})::DATE, MAX({{C_DATE}})::DATE) AS SPAN_DAYS,
    SUM(IFF({{C_DATE}}::DATE > CURRENT_DATE(), 1, 0)) AS FUTURE_VISIT_ROWS,
    SUM(IFF({{C_DATE}}::DATE < DATEADD('year', -20, CURRENT_DATE()), 1, 0)) AS VISITS_OLDER_THAN_20_YEARS
FROM {{T}};

In [ ]:
SELECT
    YEAR({{C_DATE}}) AS VISIT_YEAR,
    COUNT(*) AS VISIT_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_VISITS
FROM {{T}}
GROUP BY 1
ORDER BY 1;

In [ ]:
SELECT
    TO_CHAR({{C_DATE}}, 'YYYY-MM') AS VISIT_YEAR_MONTH,
    COUNT(*) AS VISIT_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_VISITS
FROM {{T}}
GROUP BY 1
ORDER BY 1;

In [ ]:
SELECT
    MONTH({{C_DATE}}) AS MONTH_NUM,
    TO_CHAR({{C_DATE}}, 'MON') AS MONTH_NAME,
    COUNT(*) AS VISIT_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_VISITS
FROM {{T}}
GROUP BY 1, 2
ORDER BY 1;

In [ ]:
SELECT
    DAYOFWEEKISO({{C_DATE}}) AS DOW_NUM,
    DAYNAME({{C_DATE}}) AS DAY_NAME,
    COUNT(*) AS VISIT_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_VISITS
FROM {{T}}
GROUP BY 1, 2
ORDER BY 1;

In [ ]:
WITH bucketed AS (
    SELECT
        CASE
            WHEN {{C_DATE}} IS NULL THEN 90
            WHEN {{C_DATE}}::DATE > CURRENT_DATE() THEN 80
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 30 THEN 1
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 90 THEN 2
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 180 THEN 3
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 365 THEN 4
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 2 THEN 5
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 5 THEN 6
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 7 THEN 7
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 10 THEN 8
            ELSE 9
        END AS SORT_ORDER,
        {{C_PT}} AS PATIENT_ID
    FROM {{T}}
)
SELECT
    SORT_ORDER,
    CASE SORT_ORDER
        WHEN 1 THEN '0-30 days ago'
        WHEN 2 THEN '31-90 days ago'
        WHEN 3 THEN '91-180 days ago'
        WHEN 4 THEN '181-365 days ago'
        WHEN 5 THEN '1-2 years ago'
        WHEN 6 THEN '3-5 years ago'
        WHEN 7 THEN '6-7 years ago'
        WHEN 8 THEN '8-10 years ago'
        WHEN 9 THEN 'More than 10 years ago'
        WHEN 80 THEN 'Future (after today)'
        ELSE 'Unknown (missing date)'
    END AS DATE_RANGE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM bucketed
GROUP BY 1, 2
ORDER BY SORT_ORDER;

## 8. Type and Status — unique values and counts

`Type` is text. Group by the stored string. We do not guess outpatient vs inpatient.

In [ ]:
SELECT
    {{C_TYPE}} AS VISIT_TYPE,
    COUNT(*) AS VISIT_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_VISITS
FROM {{T}}
GROUP BY 1
ORDER BY VISIT_COUNT DESC;

In [ ]:
SELECT
    {{C_STATUS}} AS STATUS,
    COUNT(*) AS VISIT_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_VISITS
FROM {{T}}
GROUP BY 1
ORDER BY VISIT_COUNT DESC;

In [ ]:
SELECT
    {{C_TYPE}} AS VISIT_TYPE,
    {{C_STATUS}} AS STATUS,
    COUNT(*) AS VISIT_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_VISITS
FROM {{T}}
GROUP BY 1, 2
ORDER BY VISIT_COUNT DESC;

## 9. Facilities and physicians

In [ ]:
SELECT
    {{C_FAC}} AS FACILITY_ID,
    COUNT(*) AS VISIT_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_PHY}}) AS UNIQUE_PHYSICIANS
FROM {{T}}
GROUP BY 1
ORDER BY VISIT_COUNT DESC
LIMIT 25;

In [ ]:
SELECT
    {{C_PHY}} AS PHYSICIAN_ID,
    COUNT(*) AS VISIT_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_FAC}}) AS UNIQUE_FACILITIES
FROM {{T}}
GROUP BY 1
ORDER BY VISIT_COUNT DESC
LIMIT 25;

## 10. Visits per patient

In [ ]:
WITH per_patient AS (
    SELECT
        {{C_PT}} AS PATIENT_ID,
        COUNT(*) AS VISIT_COUNT
    FROM {{T}}
    WHERE {{C_PT}} IS NOT NULL
    GROUP BY 1
)
SELECT
    VISIT_COUNT AS VISITS_PER_PATIENT,
    COUNT(*) AS NUMBER_OF_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_PATIENTS
FROM per_patient
GROUP BY 1
ORDER BY 1;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    COUNT(*) AS VISIT_COUNT,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_FAC}}) AS UNIQUE_FACILITIES,
    MIN({{C_DATE}}) AS FIRST_VISIT_DATE,
    MAX({{C_DATE}}) AS LAST_VISIT_DATE
FROM {{T}}
WHERE {{C_PT}} IS NOT NULL
GROUP BY 1
ORDER BY VISIT_COUNT DESC
LIMIT 25;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    {{C_ENC}} AS ENCOUNTER_ID,
    {{C_DATE}} AS VISIT_DATE,
    {{C_TYPE}} AS VISIT_TYPE,
    {{C_STATUS}} AS STATUS,
    {{C_FAC}} AS FACILITY_ID,
    {{C_PHY}} AS PHYSICIAN_ID
FROM {{T}}
WHERE {{C_PT}} = (
        SELECT {{C_PT}}
        FROM {{T}}
        WHERE {{C_PT}} IS NOT NULL
        GROUP BY 1
        ORDER BY COUNT(*) DESC
        LIMIT 1
      )
ORDER BY {{C_DATE}} NULLS LAST, {{C_ENC}};

## Notes

- **Downloading:** run a SQL cell, then use the download arrow on that result grid.
- Duplicate encounter ids are a data-quality issue; duplicate patient ids are expected.
- Run `config` before the SQL cells.